In [43]:
import pandas as pd

data = pd.read_csv("dataset/customer_shopping_behavior.csv")

In [44]:
data.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [45]:
data.columns = data.columns.str.lower()
data.columns = data.columns.str.replace(' ','_')

In [63]:
data = data.rename(columns={'purchase_amount_(usd)': 'purchase_amount_usd'})

In [64]:
data["review_rating"] = data.groupby("category")["review_rating"].transform(lambda x: x.fillna(x.median()))

In [65]:
data.isnull().sum()

customer_id                  0
age                          0
gender                       0
item_purchased               0
category                     0
purchase_amount_usd          0
location                     0
size                         0
color                        0
season                       0
review_rating                0
subscription_status          0
shipping_type                0
discount_applied             0
previous_purchases           0
payment_method               0
frequency_of_purchases       0
age_group                    0
purchase_days_frequency    584
dtype: int64

In [66]:
# Age groups
labels = ['Young', 'Adult', 'Middle-age', 'Senior']
data['age_group'] = pd.qcut(data['age'], q=4, labels = labels)

In [67]:
data[['age', 'age_group']]

,age,age_group
0,55,Middle-age
1,19,Young
2,50,Middle-age
3,21,Young
4,45,Middle-age
...,...,...
3895,40,Adult
3896,52,Middle-age
3897,46,Middle-age
3898,44,Adult


In [68]:
frequency_map = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Month': 90
}

data['purchase_days_frequency'] = data["frequency_of_purchases"].map(frequency_map)

In [69]:
data[['purchase_days_frequency', 'frequency_of_purchases']]

,purchase_days_frequency,frequency_of_purchases
0,14.0,Fortnightly
1,14.0,Fortnightly
2,7.0,Weekly
3,7.0,Weekly
4,365.0,Annually
...,...,...
3895,7.0,Weekly
3896,14.0,Bi-Weekly
3897,90.0,Quarterly
3898,7.0,Weekly


In [53]:
data[['discount_applied', 'promo_code_used']]

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
...,...,...
3895,No,No
3896,No,No
3897,No,No
3898,No,No


In [70]:
(data['discount_applied'] == data['promo_code_used']).all()

KeyError: 'promo_code_used'

In [71]:
data = data.drop('promo_code_used', axis=1)

KeyError: "['promo_code_used'] not found in axis"

In [72]:
from sqlalchemy import create_engine

# Step 1: Connect to PostgreSQL
# Replace placeholders with your actual details
username = "postgres"      # default user
password = "2096" # the password you set during installation
host = "localhost"         # if running locally
port = "5432"              # default PostgreSQL port
database = "customer_behavior"    # the database you created in pgAdmin

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# Step 2: Load DataFrame into PostgreSQL
table_name = "customer"   # choose any table name
data.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Data successfully loaded into table 'customer' in database 'customer_behavior'.
